In [ ]:
# 공통 경로와 패키지
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt
from scipy.stats import scoreatpercentile
from tqdm import tqdm
import time

AMC_INPUT_PATH = str(PROCESSED_DIR / 'amc_tidy.csv')

amc_df = pd.read_csv(AMC_INPUT_PATH, encoding= 'utf-8', dtype='str')

SMC_INPUT_PATH = str(RAW_DIR / 'D2024_0087_02(op_id 포함).csv')

smc_df = pd.read_csv(SMC_INPUT_PATH, encoding= 'utf-8', dtype='str')
smc_df['row_no'] = smc_df['row_no'].astype('int')
smc_df = smc_df.sort_values('row_no').reset_index(drop=True)

def process_age(age_str):
    age_str = str(age_str).strip()
    if '이상' in age_str:
        return 80
    elif '-' in age_str:
        return 20
    elif '대' in age_str:
        return int(age_str.replace('대', ''))
    return 0

smc_df['age'] = smc_df['age'].apply(process_age).astype(int)
smc_df = smc_df.drop(['op_id'], axis =1)


In [ ]:
# SMC 전처리
MIN_PROCEDURE_COUNT = 50
value_counts = smc_df['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= MIN_PROCEDURE_COUNT].index
smc_df = smc_df[smc_df['procedure_cd_fin'].isin(frequent_codes)]

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
smc_df = smc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
smc_df = smc_df.astype(dtype_float)

# 모델 입력 변수
MODEL_COLUMNS = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'

smc_df = smc_df[MODEL_COLUMNS]

feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

feature_count = len(feature_columns)


In [ ]:
# IQR 기준 계산

TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

# 0을 포함해 IQR 계산
lower_quartile = smc_df[TARGET_COLUMN].quantile(0.25)
upper_quartile = smc_df[TARGET_COLUMN].quantile(0.75)
interquartile_range = upper_quartile - lower_quartile

iqr_upper_bound = upper_quartile + 1.5 * interquartile_range

exceeds_iqr_limit = (smc_df[TARGET_COLUMN] > iqr_upper_bound)

exceeds_msbos_limit = (smc_df[TARGET_COLUMN] > 2 * smc_df[MSBOS_COLUMN])

# IQR 상한과 MSBOS 2배를 모두 넘으면 제외

outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

# 이상치 제외

outlier_count = outlier_mask.sum()

smc_filtered_df = smc_df[~outlier_mask].copy()

print(f"\n[INFO] 제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {outlier_count} 개")
smc_filtered_df['institution'] = 'smc'


In [ ]:
# AMC 전처리
amc_df.columns
amc_df = amc_df.drop(['pt_id', 'op_id', 'anesthesia_cd', 'anesthesia_nm2'], axis=1).rename(columns = {'anesthesia_cd2':'anesthesia_cd',
                                                                                                             'use_quan_imputed':'use_quan'})

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
amc_df = amc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
amc_df = amc_df.astype(dtype_float)

MODEL_COLUMNS = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'

amc_df = amc_df[MODEL_COLUMNS]

feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

feature_count = len(feature_columns)
amc_df.info()


In [ ]:
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

lower_quartile = amc_df[TARGET_COLUMN].quantile(0.25)
upper_quartile = amc_df[TARGET_COLUMN].quantile(0.75)
interquartile_range = upper_quartile - lower_quartile

iqr_upper_bound = upper_quartile + 1.5 * interquartile_range

exceeds_iqr_limit = (amc_df[TARGET_COLUMN] > iqr_upper_bound)

exceeds_msbos_limit = (amc_df[TARGET_COLUMN] > 2 * amc_df[MSBOS_COLUMN])

outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

outlier_count = outlier_mask.sum()

amc_filtered_df = amc_df[~outlier_mask].copy()

print(f"\n[INFO] 제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {outlier_count} 개")
amc_filtered_df['institution'] = 'amc'


In [ ]:
pd.concat([smc_filtered_df, amc_filtered_df]).reset_index(drop= 1).to_csv(str(OUTPUT_DIR / 'integration_data.csv'))
